# IGS — Indice de Gouvernance Sociale

**KPI 4 du framework Human Capital Valuation CACEIS · Alberthon 2026**

Score sur 100 mesurant la **résilience sociale** de CACEIS face aux risques régulatoires, humains et réputationnels (pilier S de l'ESG).

```
IGS = moyenne des composantes disponibles parmi  [ Mixité · Inclusion · Engagement ]
```

## Principe directeur

Calcul **rigoureusement aligné par année** : chaque année utilise **strictement** ses propres sources. Si une composante n'est pas disponible pour une année, elle est marquée **N/A** et l'IGS est calculé sur les composantes disponibles uniquement.

| Composante | 2023 | 2024 | 2025 |
|---|---:|---:|---:|
| Mixité (pay gap + femmes mgt) | ✓ | ✓ | N/A (Bilan Social 2025 non publié) |
| Inclusion (Baromètre D&I) | N/A (pas de Baromètre 2023) | N/A (pas de Baromètre 2024) | ✓ (édition 2025) |
| Engagement (intensité programmes) | ✓ | ✓ | ✓ |

## Workflow

```
1. KPIs_computations.ipynb  →  dashboard_kpi_hr.csv  (7 KPIs métier)
2. Ce notebook              →  enrichit avec igs (4 colonnes) + recalcule chhi_index_100
3. app.py                   →  affiche 4 KPIs (HCVA · KTI · RE · IGS) + radar 4 axes
```

## Pondération du CHHI

Le CHHI passe de 3 à 4 composantes :

| KPI | Poids ancien | Poids nouveau |
|---|---:|---:|
| HCVA | 40 % | **30 %** |
| KTI | 30 % | **25 %** |
| RE-Score | 30 % | **25 %** |
| **IGS** (nouveau) | — | **20 %** |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# CACEIS palette
STEEL = "#4A6E8C"; RED = "#D65063"; BLUE = "#5C768D"
GREY  = "#888B8D"; GREEN = "#2E8B57"; AMBER = "#E0A526"
DARK  = "#1F2937"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlecolor": STEEL,
})

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Cible IGS (zone verte) — utilisée pour le radar du dashboard
IGS_TARGET = 75

## 2. Inputs — données réelles extraites des documents CACEIS

**Toutes les valeurs sont calculées par nous** depuis des tableaux bruts. Aucune valeur publiée pré-calculée n'est utilisée tel quel (pas d'Index Égapro).

### Mixité 2023 — Bilan Social 2023 + Suivi accord 2023

```
Pay gap 2023      : H 59 005 € vs F 53 616 € → 9,13 %
Cadres encadrants : 98 F + 146 H = 244 → 40,16 % femmes
Effectif (CDI+CDD): 1 835
```

### Mixité 2024 — Bilan Social 2024 + Suivi accord 2024

```
Pay gap 2024      : H 60 360 € vs F 54 753 € → 9,29 %
Cadres encadrants : 97 F + 147 H = 244 → 39,75 % femmes
Effectif (CDI+CDD): 2 043
```

### Inclusion 2025 — Baromètre D&I 2025 (Mozaïk RH)

```
Baromètre France     : 70 % (effectif 2 043)
Baromètre Luxembourg : 64 % (effectif 1 882)
Score pondéré        : 67,1 %
```

### Engagement par année — programmes documentés

| Année | FAB'Life | Be Generous | We Care | Effectif total |
|---|---:|---:|---:|---:|
| 2023 | 1 184 participations | 6 lauréats | N/A | 1 835 |
| 2024 | 2 241 participations CACEIS | 17 lauréats | N/A | 2 043 |
| 2025 | N/A | 34 lauréats | 1 592 (1 256 chiffrés + 14 n/a × 24) | 3 925 (FR+Lux) |

In [ ]:
# ============================================================
# DONNÉES BRUTES PAR ANNÉE — sources strictement alignées
# ============================================================

YEARS = [2023, 2024, 2025]

INPUTS = {
    2023: {
        # Mixité — Bilan Social 2023 §2.1.2.2 + Suivi accord QVT 2023
        "sab_femmes":            53_616,
        "sab_hommes":            59_005,
        "cadres_encadrants_F":   98,
        "cadres_encadrants_H":   146,
        # Inclusion — pas de Baromètre 2023
        "barometer_fr":          None,
        "barometer_lux":         None,
        # Engagement — Bilan FAB'Life 2023 + Suivi accord 2023 (Be Generous)
        "fab_life_participations":   1_184,
        "be_generous_laureats":      6,
        "we_care_total":             None,
        # Effectif
        "headcount_fr":              1_835,
        "headcount_lux":             None,   # Bilan Social Lux non disponible 2023
    },
    2024: {
        # Mixité — Bilan Social 2024 §2.1.2.2 + Suivi accord QVT 2024
        "sab_femmes":            54_753,
        "sab_hommes":            60_360,
        "cadres_encadrants_F":   97,
        "cadres_encadrants_H":   147,
        # Inclusion — pas de Baromètre 2024
        "barometer_fr":          None,
        "barometer_lux":         None,
        # Engagement — Bilan FAB'Life 2024 + Reporting Be Generous 2024 pour CASA
        "fab_life_participations":   2_241,   # CACEIS pur (hors 743 du Crédit Agricole SA)
        "be_generous_laureats":      17,      # 10 FR + 7 Lux (et non 34 du fichier 2025)
        "we_care_total":             None,
        # Effectif
        "headcount_fr":              2_043,
        "headcount_lux":             1_882,
    },
    2025: {
        # Mixité — Bilan Social 2025 non publié
        "sab_femmes":            None,
        "sab_hommes":            None,
        "cadres_encadrants_F":   None,
        "cadres_encadrants_H":   None,
        # Inclusion — Baromètre D&I 2025 (Mozaïk RH)
        "barometer_fr":          70.0,
        "barometer_lux":         64.0,
        # Engagement — We Care 2025 + Bilan Be Generous 2025 (FAB'Life 2025 non disponible)
        "fab_life_participations":   None,
        "be_generous_laureats":      34,
        "we_care_total":             1_592,   # 1 256 chiffrés + 14 n/a × 24
        # Effectif (on garde l\'effectif 2024 puisque le Bilan Social 2025 n\'est pas publié)
        "headcount_fr":              2_043,
        "headcount_lux":             1_882,
    },
}

## 3. Fonctions de calcul des 3 composantes

Chaque fonction retourne `None` si les données ne suffisent pas pour calculer la composante cette année-là.

In [ ]:
def compute_mixite(d):
    """Score Mixité = moyenne (pay_gap_score + femmes_mgt_score)."""
    if d["sab_femmes"] is None or d["cadres_encadrants_F"] is None:
        return None
    gap_pct = (d["sab_hommes"] - d["sab_femmes"]) / d["sab_hommes"] * 100
    score_paygap = max(0, 100 - abs(gap_pct) * 5)

    total_enc = d["cadres_encadrants_F"] + d["cadres_encadrants_H"]
    pct_femmes_mgt = d["cadres_encadrants_F"] / total_enc * 100
    score_mgt = min(100, pct_femmes_mgt / 40 * 100)

    return (score_paygap + score_mgt) / 2


def compute_inclusion(d):
    """Score Inclusion = moyenne pondérée par effectif des Baromètres D&I FR + Lux."""
    if d["barometer_fr"] is None or d["barometer_lux"] is None:
        return None
    return (d["barometer_fr"]  * d["headcount_fr"] +
            d["barometer_lux"] * d["headcount_lux"]) / (d["headcount_fr"] + d["headcount_lux"])


def compute_engagement(d):
    """Score Engagement = intensité de participation × 50.
    intensité = participations totales / effectif. 1 part./employé/an = score 50.
    """
    parts = 0
    n_sources = 0
    for k in ("fab_life_participations", "be_generous_laureats", "we_care_total"):
        if d[k] is not None:
            parts += d[k]
            n_sources += 1
    if n_sources == 0:
        return None
    effectif = d["headcount_fr"] + (d["headcount_lux"] or 0)
    intensite = parts / effectif
    return min(100, intensite * 50)


def compute_igs(d):
    """IGS = moyenne arithmétique des composantes disponibles."""
    parts = []
    detail = {}
    for name, fn in [("Mixité", compute_mixite),
                     ("Inclusion", compute_inclusion),
                     ("Engagement", compute_engagement)]:
        v = fn(d)
        detail[name] = v
        if v is not None:
            parts.append(v)
    igs = sum(parts) / len(parts) if parts else None
    return igs, detail

## 4. Calcul de l'IGS pour chaque année

In [ ]:
results = []
for year in YEARS:
    d = INPUTS[year]
    igs, detail = compute_igs(d)
    row = {
        "year":       year,
        "Mixité":     round(detail["Mixité"], 1) if detail["Mixité"] is not None else None,
        "Inclusion":  round(detail["Inclusion"], 1) if detail["Inclusion"] is not None else None,
        "Engagement": round(detail["Engagement"], 1) if detail["Engagement"] is not None else None,
        "IGS":        round(igs, 1) if igs is not None else None,
    }
    results.append(row)

igs_df = pd.DataFrame(results).set_index("year")
print(igs_df.fillna("N/A"))

## 5. Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)

x = np.arange(len(YEARS))
w = 0.21

mixite_vals      = [r["Mixité"]     if r["Mixité"]     is not None else 0 for r in results]
inclusion_vals   = [r["Inclusion"]  if r["Inclusion"]  is not None else 0 for r in results]
engagement_vals  = [r["Engagement"] if r["Engagement"] is not None else 0 for r in results]
igs_vals         = [r["IGS"]        if r["IGS"]        is not None else 0 for r in results]

ax.bar(x - 1.5*w, mixite_vals,     w, label="Mixité",     color=BLUE,  edgecolor="white")
ax.bar(x - 0.5*w, inclusion_vals,  w, label="Inclusion",  color=AMBER, edgecolor="white")
ax.bar(x + 0.5*w, engagement_vals, w, label="Engagement", color=GREY,  edgecolor="white")
ax.bar(x + 1.5*w, igs_vals,        w, label="IGS final",  color=RED,   edgecolor="white")

# Marquer les N/A
for i, r in enumerate(results):
    for j, (key, vals) in enumerate(zip(("Mixité","Inclusion","Engagement"),
                                         [mixite_vals, inclusion_vals, engagement_vals])):
        if r[key] is None:
            ax.text(x[i] + (j-1.5)*w + 0.05, 2, "N/A", fontsize=9, color=GREY,
                    rotation=90, va="bottom")

ax.axhline(IGS_TARGET, color=GREEN, linestyle="--", linewidth=1, alpha=0.6)
ax.text(len(YEARS)-0.5, IGS_TARGET+1.5, f"Cible {IGS_TARGET}", color=GREEN, fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(YEARS)
ax.set_ylabel("Score (/100)")
ax.set_ylim(0, 100)
ax.set_title("IGS — Composantes et score consolidé par année", fontsize=13, loc="left", pad=10)
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#E5E9EE", linewidth=0.6); ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "igs_dashboard.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

## 6. Export — enrichissement de `dashboard_kpi_hr.csv`

On ajoute 4 colonnes (`igs`, `igs_mixite`, `igs_inclusion`, `igs_engagement`) ainsi qu'`igs_radar` (achievement vs cible 75) pour le dashboard Streamlit.

Le CHHI est aussi recalculé avec la nouvelle pondération **30 / 25 / 25 / 20** (HCVA · KTI · RE · IGS).

In [ ]:
# Trace audit indépendante
audit_path = OUTPUT_DIR / "igs_yearly.csv"
igs_df.to_csv(audit_path, sep=";", encoding="utf-8-sig")
print(f"✓ Audit : {audit_path}")
print()

# Enrichissement du CSV central
central_csv = "dashboard_kpi_hr.csv"
if os.path.exists(central_csv):
    central = pd.read_csv(central_csv, sep=";")

    # Ajout des colonnes IGS
    igs_by_year = {y: r for y, r in zip(YEARS, results)}
    central["igs"]            = central["year"].map(lambda y: igs_by_year.get(int(y), {}).get("IGS",        0) or 0)
    central["igs_mixite"]     = central["year"].map(lambda y: igs_by_year.get(int(y), {}).get("Mixité",     0) or 0)
    central["igs_inclusion"]  = central["year"].map(lambda y: igs_by_year.get(int(y), {}).get("Inclusion",  0) or 0)
    central["igs_engagement"] = central["year"].map(lambda y: igs_by_year.get(int(y), {}).get("Engagement", 0) or 0)

    # Radar achievement % vs cible IGS=75
    central["igs_radar"] = (central["igs"] / IGS_TARGET * 100).round(2)

    # Recalcul du CHHI avec IGS intégré (pondération 30/25/25/20)
    if all(c in central.columns for c in ["hcva_radar", "kti_radar", "re_radar"]):
        central["chhi_index_100"] = (
            0.30 * central["hcva_radar"] +
            0.25 * central["kti_radar"]  +
            0.25 * central["re_radar"]   +
            0.20 * central["igs_radar"]
        ).round(2)
        print("✓ CHHI recalculé avec IGS intégré (pondération 30/25/25/20)")

    central.to_csv(central_csv, index=False, sep=";", encoding="utf-8-sig")
    print(f"✓ {central_csv} enrichi avec IGS et CHHI mis à jour")
    print()
    cols_show = ["year", "igs_mixite", "igs_inclusion", "igs_engagement", "igs", "igs_radar", "chhi_index_100"]
    cols_show = [c for c in cols_show if c in central.columns]
    print(central[cols_show])
else:
    print(f"⚠  {central_csv} introuvable — lance d\'abord KPIs_computations.ipynb.")
    print("   La trace audit outputs/igs_yearly.csv reste disponible.")

## 7. Pitch board en 30 secondes

> *« CACEIS gouvernance sociale — IGS sur 3 ans :*
>
> *2023 : 54,8 (zone faible) — Mixité OK, Engagement encore en construction.*
> *2024 : 65,9 (zone jaune) — Mixité maintenue, Engagement en croissance forte (+22 pts).*
> *2025 : 43,9 (zone faible) — Inclusion mesurée pour la 1ʳᵉ fois (Baromètre D&I), Engagement plus diversifié mais moins intense.*
>
> *La Mixité est notre force structurelle. L'Engagement progresse. L'Inclusion mérite d'être mesurée chaque année (pas seulement en édition ponctuelle). »*

---

## 8. Sources

| Année | Mixité | Inclusion | Engagement |
|---|---|---|---|
| 2023 | `2023 _Bilan Social VF.pdf` §2.1.2.2 + `2023 _Suivi accord QVT VF.pdf` | — | `2023 _BILAN FAB'Life programme.pptx` + lauréats Be Generous (Suivi accord) |
| 2024 | `2024 _Bilan Social.pdf` §2.1.2.2 + `2024 _Suivi accord QVT vDef.pdf` | — | `2024 _Bilan FAB'Life.pptx` + `2024 _Reporting Be Generous pour CASA.xlsx` |
| 2025 | — | `2025_Baromètre D&I CACEIS - France/Luxembourg.pdf` | `2025 _We Care - Bilan.xlsx` + `2025 _Bilan Groupe Be Generous CACEIS.xlsx` |

**Standards :** ISO 30414 · ESRS S1 (CSRD) · AI Act Art. 10 · Loi Rixain